In [1]:
import torch
import pandas as pd
from question_loader import QuestionDataset
from pathlib import Path
import numpy as np

dataset = QuestionDataset("../question_database/preprocess")
df = pd.DataFrame([dataset[i] for i in range(len(dataset))])
df[["dataset"]].value_counts()

OUT_DIR = Path("../question_database/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUT_DIR / "questions_master.parquet", index=False)
df.to_csv(OUT_DIR / "questions_master.csv", index=False)


[QuestionDataset] Loaded 145 questions from 6 files


In [2]:
import pandas as pd
import numpy as np

path = "../question_database/raw/HUMAN.xlsx"
cols_to_drop = ["ID", "Start time", "Completion time", "Email"]

# load
df_raw = pd.read_excel(path, engine="openpyxl")

# drop metadata columns
df_raw = df_raw.drop(columns=cols_to_drop, errors="ignore")

# drop columns with any NaN (incomplete annotations)
df_raw = df_raw.dropna(axis=1, how="any")

print("Raw shape (respondents and questions):", df_raw.shape)

# build question-centric table
human_long = pd.DataFrame({
    "row_id": np.arange(len(df_raw.columns)),
    "question_text": df_raw.columns.tolist(),
    "human_answers": [
        list(dict.fromkeys(df_raw[col].astype(str).tolist()))
        for col in df_raw.columns
    ]
})

display(human_long.head())

Raw shape (respondents and questions): (28, 179)


,row_id,question_text,human_answers
0,0,I am able to adapt when changes occur.,"[Emotional, Social, Environmental, Intellectual]"
1,1,I have one close and secure relationship.,"[Social, Emotional]"
2,2,Sometimes fate or God helps me.,[Spiritual]
3,3,I can deal with whatever comes my way.,"[Emotional, Spiritual, Intellectual, Occupatio..."
4,4,Past successes give me confidence.,"[Emotional, Intellectual, Occupational, Enviro..."


In [3]:
text2qid = dict(zip(df["text"], df["qid"]))


human_long["qid"] = human_long["question_text"].map(text2qid)


human_long = human_long.drop(columns=["row_id"], errors="ignore")
cols = ["qid"] + [c for c in human_long.columns if c != "qid"]
human_long = human_long[cols]
human_long = human_long.dropna(subset=["qid"])
display(human_long.head())
human_long.to_csv(OUT_DIR / "human_answer.csv", index=False)


,qid,question_text,human_answers
0,CD_RISC_1,I am able to adapt when changes occur.,"[Emotional, Social, Environmental, Intellectual]"
1,CD_RISC_2,I have one close and secure relationship.,"[Social, Emotional]"
2,CD_RISC_3,Sometimes fate or God helps me.,[Spiritual]
3,CD_RISC_4,I can deal with whatever comes my way.,"[Emotional, Spiritual, Intellectual, Occupatio..."
4,CD_RISC_5,Past successes give me confidence.,"[Emotional, Intellectual, Occupational, Enviro..."


In [ ]:
import pandas as pd
import numpy as np

path = "../question_database/raw/HUMAN.xlsx"
cols_to_drop = ["ID", "Start time", "Completion time", "Email"]

df_raw = pd.read_excel(path, engine="openpyxl")
df_raw = df_raw.drop(columns=cols_to_drop, errors="ignore")

print("Raw shape (respondents and questions):", df_raw.shape)

q_meta = pd.DataFrame({"text": df_raw.columns.tolist()})
q_meta["qid"] = q_meta["text"].map(text2qid)

q_meta = q_meta.dropna(subset=["qid"]).reset_index(drop=True)
df_raw = df_raw[q_meta["text"].tolist()]

n_resp = df_raw.shape[0]
resp_ids = [f"human_{i:02d}" for i in range(1, n_resp + 1)]
df_raw.index = resp_ids

human_long = (
    df_raw.reset_index(names="dimension_model")
         .melt(id_vars="dimension_model", var_name="text", value_name="answer")
         .merge(q_meta, on="text", how="left")
)

# drop missing answers only (not whole respondents)
human_long = human_long.dropna(subset=["answer"])

human_long["dataset"] = "HUMAN"
human_long["answer"] = human_long["answer"].astype(str).str.strip()
human_long["dimensions"] = human_long["answer"].apply(lambda x: [x] if x else [])

human_long = human_long[["qid", "dataset", "text", "dimension_model", "dimensions"]]

display(human_long.head())
human_long.to_csv(OUT_DIR / "human_all_results_like.csv", index=False)

Raw shape (respondents and questions): (28, 562)


,qid,dataset,text,dimension_model,dimensions
0,CD_RISC_1,HUMAN,I am able to adapt when changes occur.,human_01,[Emotional]
1,CD_RISC_1,HUMAN,I am able to adapt when changes occur.,human_02,[Emotional]
2,CD_RISC_1,HUMAN,I am able to adapt when changes occur.,human_03,[Social]
3,CD_RISC_1,HUMAN,I am able to adapt when changes occur.,human_04,[Emotional]
4,CD_RISC_1,HUMAN,I am able to adapt when changes occur.,human_05,[Environmental]
